In [1]:
import pandas as pd
print("Done.")
'''
gdpo=pd.read_csv('GDP.csv',  skiprows=4)
gdpo.head() 
inf=pd.read_csv('Inflation.csv',  skiprows=4)
inf.head() '''

Done.


"\ngdpo=pd.read_csv('GDP.csv',  skiprows=4)\ngdpo.head() \ninf=pd.read_csv('Inflation.csv',  skiprows=4)\ninf.head() "

In [3]:
inflation_raw=pd.read_csv("data/raw/Inflation.csv", skiprows=4)
inflation_pk=inflation_raw[inflation_raw['Country Code']=='PAK'].copy()
years = [str(y) for y in range(2014, 2025)]
inflation_pk = inflation_pk[years]
inflation = inflation_pk.T.reset_index()
inflation.columns = ['year', 'inflation_rate']
inflation['year'] = inflation['year'].astype(int)
print(inflation)


    year  inflation_rate
0   2014        7.189384
1   2015        2.529328
2   2016        3.765119
3   2017        4.085374
4   2018        5.078057
5   2019       10.578031
6   2020        9.740116
7   2021        9.496416
8   2022       19.873860
9   2023       30.768128
10  2024       12.632532


In [4]:
gdp_raw=pd.read_csv('data/raw/GDP.csv', skiprows=4)
gdp_pk=gdp_raw[gdp_raw['Country Code']=='PAK'].copy()
gdp_pk=gdp_pk[years]
gdp=gdp_pk.T.reset_index()
gdp.columns=['year','usd']
gdp['year']=gdp['year'].astype(int)
gdp['gdp_billions'] = gdp['usd'] / 1e9
print(gdp)

    year           usd  gdp_billions
0   2014  2.713905e+11    271.390475
1   2015  2.999636e+11    299.963591
2   2016  3.136300e+11    313.630000
3   2017  3.392055e+11    339.205535
4   2018  3.561282e+11    356.128167
5   2019  3.209095e+11    320.909473
6   2020  3.004256e+11    300.425610
7   2021  3.485166e+11    348.516647
8   2022  3.748903e+11    374.890296
9   2023  3.366863e+11    336.686349
10  2024  3.715700e+11    371.570000


In [5]:
minwage_raw = pd.read_csv('data/raw/Minimum Wage.csv')


minwage = minwage_raw[minwage_raw['classif1.label'] == 'Currency: Local currency'].copy()
minwage = minwage[['time', 'obs_value']].copy()
minwage.columns = ['year', 'min_wage_pkr']
minwage['year'] = minwage['year'].astype(int)

minwage = minwage[minwage['year'].between(2014, 2024)]

print(minwage)

    year  min_wage_pkr
0   2024       37000.0
3   2023       32000.0
6   2022       25000.0
9   2021       19000.0
12  2020       17500.0
15  2019       17500.0
18  2018       15000.0
21  2017       15000.0
24  2016       14000.0
27  2015       13000.0
30  2014       10000.0


In [6]:
df = pd.merge(inflation, gdp[['year', 'gdp_billions']], on='year')
df = pd.merge(df, minwage, on='year')

df = df.sort_values('year').reset_index(drop=True)

print(df)

    year  inflation_rate  gdp_billions  min_wage_pkr
0   2014        7.189384    271.390475       10000.0
1   2015        2.529328    299.963591       13000.0
2   2016        3.765119    313.630000       14000.0
3   2017        4.085374    339.205535       15000.0
4   2018        5.078057    356.128167       15000.0
5   2019       10.578031    320.909473       17500.0
6   2020        9.740116    300.425610       17500.0
7   2021        9.496416    348.516647       19000.0
8   2022       19.873860    374.890296       25000.0
9   2023       30.768128    336.686349       32000.0
10  2024       12.632532    371.570000       37000.0


In [7]:
df['real_wage_pkr'] = df['min_wage_pkr'].copy()

base_year = df[df['year'] == 2014]['min_wage_pkr'].values[0]

for i in range(1, len(df)):
    df.loc[i, 'real_wage_pkr'] = df.loc[i-1, 'real_wage_pkr'] / (1 + df.loc[i, 'inflation_rate'] / 100)

print(df[['year', 'min_wage_pkr', 'real_wage_pkr', 'inflation_rate']])

    year  min_wage_pkr  real_wage_pkr  inflation_rate
0   2014       10000.0   10000.000000        7.189384
1   2015       13000.0    9753.306862        2.529328
2   2016       14000.0    9399.407952        3.765119
3   2017       15000.0    9030.479134        4.085374
4   2018       15000.0    8594.067467        5.078057
5   2019       17500.0    7771.948391       10.578031
6   2020       17500.0    7082.139750        9.740116
7   2021       19000.0    6467.919247        9.496416
8   2022       25000.0    5395.604387       19.873860
9   2023       32000.0    4126.085206       30.768128
10  2024       37000.0    3663.315685       12.632532


In [8]:
df['purchasing_power_loss_%'] = ((df['real_wage_pkr'] - df['min_wage_pkr']) / df['min_wage_pkr'] * 100).round(2)
print(df[['year', 'min_wage_pkr', 'real_wage_pkr', 'purchasing_power_loss_%']])

    year  min_wage_pkr  real_wage_pkr  purchasing_power_loss_%
0   2014       10000.0   10000.000000                     0.00
1   2015       13000.0    9753.306862                   -24.97
2   2016       14000.0    9399.407952                   -32.86
3   2017       15000.0    9030.479134                   -39.80
4   2018       15000.0    8594.067467                   -42.71
5   2019       17500.0    7771.948391                   -55.59
6   2020       17500.0    7082.139750                   -59.53
7   2021       19000.0    6467.919247                   -65.96
8   2022       25000.0    5395.604387                   -78.42
9   2023       32000.0    4126.085206                   -87.11
10  2024       37000.0    3663.315685                   -90.10


In [9]:
df.to_csv('pakistan_wages_analysis.csv', index=False)
